# Field Scan : DeepForest entraîné sur « Count banana plants », testé sur Manuel

**Question** : avec des annotations propres (Count banana plants v15, licence CC BY 4.0), DeepForest compte-t-il mieux
les bananiers de ReforesTree que la méthode simple ?

- **Apprentissage** : vols `100_0221` et `100_0222` de Count banana plants.
- **Validation** (choix du réglage nms / seuil) : vols `100_0223` et `100_0224`. Jamais deux morceaux d'une même photo des deux côtés.
- **Test** : ferme **Manuel** (ReforesTree), notée avec le réglage choisi sur la validation, sans rien régler sur Manuel.
- Nestor n'est pas utilisé.

**Échelle** : la taille du pixel de Count banana plants n'est pas connue. Je l'estime à 3 cm (un bananier adulte d'environ 3 m
fait environ 105 px). Les images sont réduites à 2/3 pour passer à 4,5 cm, comme mes tuiles ReforesTree.

**Version Kaggle.** Avant de lancer, dans le panneau de droite : Accelerator = **GPU T4**, Internet = **On**.
Données ajoutées en « Input » : le dataset `count_banana_v15` (zip Roboflow) et le dataset contenant `tuiles.zip`.
Kaggle dézippe tout seul ; le notebook retrouve les fichiers où qu'ils soient dans `/kaggle/input`.
Résultats : `/kaggle/working/resultats_count_banana/` (onglet Output).


In [ ]:
# 1. Installation et repérage des données dans /kaggle/input
!pip install -q "deepforest>=2,<3"
from pathlib import Path
import torch
ENTREE = Path('/kaggle/input')
coco = sorted(ENTREE.rglob('_annotations.coco.json'))
tuiles_csv = sorted(ENTREE.rglob('tuiles.csv'))
print(len(coco), 'fichiers COCO trouvés (attendu : 3)')
print('tuiles.csv :', tuiles_csv)
assert len(coco) == 3 and len(tuiles_csv) == 1, "Données manquantes : vérifier les deux datasets en Input"
print('GPU :', torch.cuda.is_available())


In [ ]:
# 2. Préparation de Count banana plants : réduction 3 cm -> 4,5 cm, boîtes au format DeepForest, vol de chaque image
import json, re, time
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image

CB = Path('/kaggle/working/count_banana'); (CB / 'images').mkdir(parents=True, exist_ok=True)
PIXEL_ORIGINE, PIXEL_CIBLE = 0.03, 0.045          # 0,03 m = estimation
F = PIXEL_ORIGINE / PIXEL_CIBLE
LABEL = 'Tree'

boites, images = [], []
for fichier in coco:
    j = json.load(open(fichier)); par_image = {}
    for a in j['annotations']:
        par_image.setdefault(a['image_id'], []).append(a['bbox'])        # bbox COCO = x, y, largeur, hauteur
    for im in j['images']:
        nom = im['file_name'].split('_JPG')[0] + '.png'
        img = Image.open(fichier.parent / im['file_name']).convert('RGB')
        img.resize((round(img.width * F), round(img.height * F)), Image.Resampling.LANCZOS).save(CB / 'images' / nom)
        images.append({'image_path': nom, 'vol': re.match(r'(100_\d+)', nom)[1]})
        for x, y, w, h in par_image.get(im['id'], []):
            if w * F >= 2 and h * F >= 2:
                boites.append({'image_path': nom, 'xmin': round(x * F), 'ymin': round(y * F),
                               'xmax': round((x + w) * F), 'ymax': round((y + h) * F), 'label': LABEL})
boites, images = pd.DataFrame(boites), pd.DataFrame(images)
print(len(images), 'images,', len(boites), 'bananiers')
print(images.vol.value_counts())

VOLS_APPRENTISSAGE = ['100_0221', '100_0222']
VOLS_VALIDATION = ['100_0223', '100_0224']
app = images[images.vol.isin(VOLS_APPRENTISSAGE)].image_path
val = images[images.vol.isin(VOLS_VALIDATION)].image_path
boites[boites.image_path.isin(app)].to_csv(CB / 'apprentissage.csv', index=False)
print(len(app), 'images apprentissage |', len(val), 'images validation')


In [ ]:
# 3. Outils de notation (règle du sujet : centre à moins de 1 m d'un vrai bananier)
from scipy.optimize import linear_sum_assignment

RAYON_M = 1.0
SEUILS_CONFIANCE = np.round(np.arange(0.05, 0.96, 0.05), 2)
SEUILS_NMS = [0.05, 0.2, 0.4]

def centres(df):
    return np.c_[(df.xmin + df.xmax) / 2, (df.ymin + df.ymax) / 2]

def apparier(det, vrai, rayon_px):
    if len(det) == 0 or len(vrai) == 0:
        return 0
    d = np.linalg.norm(det[:, None] - vrai[None], axis=2)
    cout = np.where(d <= rayon_px, d, 1e6)
    l, c = linear_sum_assignment(cout)
    return int((cout[l, c] <= rayon_px).sum())

def grille(modele, chemins, verite, taille_pixel_m):
    # chemins : liste de (nom, chemin complet) ; verite : boîtes vraies (image_path, xmin...) ; taille_pixel_m : un nombre par image
    lignes = []
    for nms in SEUILS_NMS:
        modele.model.nms_thresh = nms
        modele.model.score_thresh = 0.05
        pred = {}
        for nom, chemin in chemins:
            p = modele.predict_image(path=str(chemin))
            pred[nom] = p if p is not None else pd.DataFrame(columns=['xmin', 'ymin', 'xmax', 'ymax', 'score'])
        for s in SEUILS_CONFIANCE:
            vp = nb = 0
            for nom, _ in chemins:
                p = pred[nom]; p = p[p.score >= s]
                vp += apparier(centres(p), centres(verite[verite.image_path == nom]), RAYON_M / taille_pixel_m[nom])
                nb += len(p)
            lignes.append({'nms': nms, 'seuil': s, 'detections': nb, 'bananes_vraies': len(verite), 'vp': vp})
    g = pd.DataFrame(lignes)
    g['precision'] = g.vp / g.detections.clip(lower=1)
    g['rappel'] = g.vp / g.bananes_vraies
    g['f1'] = 2 * g.vp / (g.detections + g.bananes_vraies)
    return g


In [ ]:
# 4. Apprentissage (modèle DeepForest d'origine « arbres », 10 époques)
from deepforest import main
try:
    from lightning.pytorch.loggers import CSVLogger
except ImportError:
    from pytorch_lightning.loggers import CSVLogger

EPOQUES = 10
SORTIE = Path('/kaggle/working/resultats_count_banana')
SORTIE.mkdir(parents=True, exist_ok=True)

torch.manual_seed(0)
modele = main.deepforest()
modele.load_model('weecology/deepforest-tree')
modele.config.train.csv_file = str(CB / 'apprentissage.csv')
modele.config.train.root_dir = str(CB / 'images')
modele.config.train.epochs = EPOQUES
modele.config.batch_size = 8
modele.config.workers = 2
journal = CSVLogger('/kaggle/working/journaux', name='count_banana')
debut = time.time()
modele.create_trainer(logger=journal)
modele.trainer.fit(modele)
print(f'Apprentissage : {(time.time() - debut) / 60:.1f} min')

# Sécurités : toutes les époques faites ET une erreur (loss) qui est un vrai nombre à chaque époque
j = pd.read_csv(Path(journal.log_dir) / 'metrics.csv')
pertes = j.dropna(subset=['train_loss_epoch']) if 'train_loss_epoch' in j else j.iloc[0:0]
if j.epoch.max() + 1 < EPOQUES or len(pertes) < EPOQUES or not np.isfinite(pertes.train_loss_epoch).all():
    raise RuntimeError("Apprentissage incomplet ou instable (loss absente ou NaN) : ne pas utiliser ce modèle.")
print(pertes[['epoch', 'train_loss_epoch']].to_string(index=False))
j.to_csv(SORTIE / 'journal.csv', index=False)
torch.save(modele.model.state_dict(), SORTIE / 'poids_count_banana.pt')


In [ ]:
# 5. Validation (vols 0223 et 0224) : on choisit nms et seuil ICI, pas sur Manuel
chemins_val = [(n, CB / 'images' / n) for n in val]
g_val = grille(modele, chemins_val, boites[boites.image_path.isin(val)], {n: PIXEL_CIBLE for n in val})
meilleur = g_val.loc[g_val.f1.idxmax()]
NMS, SEUIL = meilleur.nms, meilleur.seuil
print(f'Validation Count banana : F1 {meilleur.f1:.3f} (précision {meilleur.precision:.3f}, rappel {meilleur.rappel:.3f}) '
      f'avec nms {NMS}, seuil {SEUIL}')
g_val.to_csv(SORTIE / 'grille_validation.csv', index=False)


In [ ]:
# 6. Test sur Manuel (ReforesTree) avec le réglage choisi en 5
TUILES = tuiles_csv[0].parent
tuiles = pd.read_csv(TUILES / 'tuiles.csv')
ann = pd.read_csv(TUILES / 'annotations.csv')
tuiles['y0'] = tuiles.image_path.str.extract(r'_(\d+)_\d+\.png$')[0].astype(int)
tuiles['x0'] = tuiles.image_path.str.extract(r'_(\d+)\.png$')[0].astype(int)
manuel = tuiles[(tuiles.site == 'Manuel Macias RGB') & (tuiles.y0 % 400 == 0) & (tuiles.x0 % 400 == 0)]   # sans chevauchement
verite_manuel = ann[(ann.label == 'banane') & ann.image_path.isin(manuel.image_path)]
g_test = grille(modele, [(n, TUILES / 'images' / n) for n in manuel.image_path], verite_manuel,
                dict(zip(manuel.image_path, manuel.taille_pixel_m)))
test = g_test[(g_test.nms == NMS) & (g_test.seuil == SEUIL)].iloc[0]
print(f'MANUEL, réglage choisi sur la validation : F1 {test.f1:.3f} '
      f'(précision {test.precision:.3f}, rappel {test.rappel:.3f}, {int(test.detections)} détections pour {int(test.bananes_vraies)} bananiers)')
print(f'Pour info, meilleur F1 possible sur Manuel (réglage choisi sur Manuel, donc optimiste) : {g_test.f1.max():.3f}')
print('Comparaison : méthode simple 0.204 | DeepForest entraîné sur ReforesTree (essai0) 0.291')
g_test.to_csv(SORTIE / 'grille_manuel.csv', index=False)
